# M05 — Generator: zero-shot check + SFT (Qwen3.5-0.8B)

Run top-to-bottom on **Google Colab** (T4 GPU, `Runtime -> Run all`). Expect **~60 min**
(models + dataset come from the Drive cache on re-runs).

Stages:

1. **Bank lock** — rebuild the frozen Option B request bank (200 requests, seed
   20260920) from the pinned pools and assert its sha256 against the local lock.
2. **Zero-shot instruction-following** — before any training, run all 200 bank
   requests through Qwen3.5-0.8B (text-only, non-thinking, greedy) and measure the
   output-contract pass rate: JSON object with a `text` field, target name exactly
   once. This is the plan's required check before spending on training.
3. **SFT** — LoRA fine-tune on the dataset-derived single-name pairs (train-side rows
   only; final-eval rows excluded; entity values -> original text as the assistant
   completion in the JSON output format). Assistant-token loss only.
4. **Post-SFT re-check** — same 200 requests through the tuned generator; the
   zero-shot vs SFT pass-rate delta is the notebook's headline number.

Everything (adapter, report, metrics, error examples) is saved to Drive under
`MyDrive/pii-redteam-rl/m05-<timestamp>/`. The frozen detector D0 is NOT used here; it
enters at M07 (reward). No final-eval data is used for training or tuning.

In [ ]:
import datetime
import json
import os
import sys

REPO_URL = "https://github.com/GauthierRoy/pii-redteam-rl.git"
REPO_DIR = "/content/pii-redteam-rl"
DRIVE_ROOT = "/content/drive/MyDrive/pii-redteam-rl"
RUN_ID = datetime.datetime.now(datetime.UTC).strftime("m05-%Y%m%d-%H%M%S")
WORK = os.path.join(DRIVE_ROOT, RUN_ID)
CACHE = os.path.join(DRIVE_ROOT, "cache")
EXPECTED_FINAL_EVAL_SHA = "513335badf5752a340698d3c7efa7029a8780c51fde2675c80ef4208ce27d39c"
EXPECTED_BANK_SHA = "332a9eb0b682bc7213aa556290d1eeb9f616be81b3146c382bcf9c37300dc58c"

SEED = 20260920
BANK_N = 200
GEN_MAX_NEW_TOKENS = 150
LORA_R = 16
EPOCHS = 3
SFT_BATCH = 8
LR = 1e-4

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True)
os.makedirs(CACHE, exist_ok=True)

REPORT = {"run_id": RUN_ID, "checks": {}, "config": {
    "seed": SEED, "bank_n": BANK_N, "max_new_tokens": GEN_MAX_NEW_TOKENS,
    "lora_r": LORA_R, "epochs": EPOCHS, "batch": SFT_BATCH, "lr": LR,
}}


def check(name, ok, detail=""):
    REPORT["checks"][name] = {"ok": bool(ok), "detail": str(detail)}
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f" -- {detail}" if detail else ""))


print(f"run_id: {RUN_ID}  ->  {WORK}")

In [ ]:
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
REPO_COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate", "peft", "torchao"],
    check=True,
)
import torch  # noqa: E402
import transformers  # noqa: E402

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from pii_redteam.data import QuarantinedRecord, build_sft_pair, is_single_person_once, to_canonical  # noqa: E402
from pii_redteam.requests import render_prompt  # noqa: E402

torch.manual_seed(SEED)
REPORT["environment"] = {
    "repo_commit": REPO_COMMIT, "python": sys.version.split()[0],
    "torch": torch.__version__, "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
}
print(json.dumps(REPORT["environment"], indent=2))
assert torch.cuda.is_available(), "GPU runtime required."

In [ ]:
import hashlib

# Rebuild the locked splits, then the frozen request bank; assert both locks.
splits_dir = os.path.join(REPO_DIR, "artifacts", "splits")
subprocess.run(
    [sys.executable, "scripts/carve_splits.py", "--cache-dir", CACHE, "--out", splits_dir],
    cwd=REPO_DIR, env=dict(os.environ, PYTHONPATH=os.path.join(REPO_DIR, "src")), check=True,
)
manifest = json.load(open(os.path.join(splits_dir, "final_eval_manifest.json")))
check("final-eval lock reproduces", manifest["final_eval_sha256"] == EXPECTED_FINAL_EVAL_SHA,
      manifest["final_eval_sha256"])
if manifest["final_eval_sha256"] != EXPECTED_FINAL_EVAL_SHA:
    raise SystemExit("Lock mismatch.")

subprocess.run(
    [sys.executable, "scripts/build_request_bank.py", "--n", str(BANK_N), "--seed", str(SEED)],
    cwd=REPO_DIR, env=dict(os.environ, PYTHONPATH=os.path.join(REPO_DIR, "src")), check=True,
)
bank_path = os.path.join(REPO_DIR, "artifacts", "request_bank", f"sampler_seed{SEED}_n{BANK_N}.jsonl")
bank_sha = hashlib.sha256(open(bank_path, "rb").read()).hexdigest()
check("request bank reproduces (frozen lock)", bank_sha == EXPECTED_BANK_SHA, bank_sha)
REQUESTS = [json.loads(line) for line in open(bank_path, encoding="utf-8")]
print(f"bank: {len(REQUESTS)} requests")

# Dataset-derived SFT pairs: single-person-once train-side rows, final-eval excluded.
FINAL_EVAL_IDS = {r["example_id"] for r in manifest["rows"]}
pairs = []
with open(os.path.join(CACHE, "1english_openpii_30k.jsonl"), encoding="utf-8") as f:
    for line in f:
        try:
            canonical = to_canonical(json.loads(line))
        except QuarantinedRecord:
            continue
        if canonical["example_id"] in FINAL_EVAL_IDS or not is_single_person_once(canonical):
            continue
        request, target = build_sft_pair(canonical, request_id=f"sft-{len(pairs):04d}")
        pairs.append({"request": request, "target": target})
print(f"SFT pairs: {len(pairs)}")
REPORT["sft_pairs"] = len(pairs)

In [ ]:
import time

from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN_ID = "Qwen/Qwen3.5-0.8B"
tok = AutoTokenizer.from_pretrained(QWEN_ID)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(QWEN_ID, dtype=dtype, device_map="cuda")
check("Qwen3.5-0.8B text-only load", True, f"AutoModelForCausalLM dtype={dtype}")


def chat_text_for(request):
    try:
        return tok.apply_chat_template(
            [{"role": "user", "content": render_prompt(request)}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
    except TypeError:
        return tok.apply_chat_template(
            [{"role": "user", "content": render_prompt(request)}],
            tokenize=False, add_generation_prompt=True,
        )


@torch.no_grad()
def generate_batch(requests, batch_size=16):
    """Greedy batched generation; returns list of decoded continuation strings."""
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(requests), batch_size):
        chunk = requests[i : i + batch_size]
        texts = [chat_text_for(r) for r in chunk]
        enc = tok(texts, return_tensors="pt", padding=True).to("cuda")
        gen = model.generate(**enc, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=False)
        for j in range(len(chunk)):
            outs.append(tok.decode(gen[j][enc["input_ids"].shape[1]:], skip_special_tokens=True))
        if (i // batch_size) % 2 == 0:
            print(f"  generated {i + len(chunk)}/{len(requests)}")
    return outs


def contract_results(requests, outputs):
    ok_json = 0
    ok_once = 0
    failures = []
    for request, out in zip(requests, outputs, strict=True):
        name = request["task"]["entities"][0]["value"]
        try:
            parsed = json.loads(out)
            text = parsed.get("text") if isinstance(parsed, dict) else None
            good_json = isinstance(text, str)
        except json.JSONDecodeError:
            text, good_json = None, False
        ok_json += good_json
        once = bool(text) and text.count(name) == 1
        ok_once += once
        if not (good_json and once) and len(failures) < 20:
            failures.append({"request_id": request["request_id"], "name": name,
                             "output": out[:400], "json_ok": good_json, "occurrences": text.count(name) if text else -1})
    return {"pass_json": ok_json, "pass_once": ok_once, "total": len(requests),
            "rate_json": round(ok_json / len(requests), 4), "rate_once": round(ok_once / len(requests), 4),
            "failures": failures}


t0 = time.time()
zeroshot_outputs = generate_batch(REQUESTS)
zeroshot_s = round(time.time() - t0, 1)
zero = contract_results(REQUESTS, zeroshot_outputs)
REPORT["zeroshot"] = {k: zero[k] for k in ("pass_json", "pass_once", "total", "rate_json", "rate_once")}
print(f"zero-shot: JSON {zero['rate_json']:.0%}, name-exactly-once {zero['rate_once']:.0%} ({zeroshot_s}s)")
check("zero-shot instruction-following measured", True,
      f"json_ok={zero['rate_json']:.2f} name_once={zero['rate_once']:.2f} (informational, no floor)")
print(json.dumps(zero["failures"][:3], indent=2, ensure_ascii=False))

In [ ]:
import time

import math

from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader, Dataset

# LoRA adapter (gate-proven path); assistant-token loss only.
linear_names = [n.split(".")[-1] for n, m in model.named_modules() if isinstance(m, torch.nn.Linear)]
targets = sorted({n for n in linear_names if n in {"q_proj", "k_proj", "v_proj", "o_proj"}}) or ["q_proj"]
model = get_peft_model(
    model,
    LoraConfig(r=LORA_R, lora_alpha=2 * LORA_R, target_modules=targets,
               lora_dropout=0.0, task_type="CAUSAL_LM"),
)
model.print_trainable_parameters()


class SftDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        item = self.items[i]
        prompt = chat_text_for(item["request"])
        answer = json.dumps({"text": item["target"]}, ensure_ascii=False) + tok.eos_token
        prompt_ids = tok(prompt, add_special_tokens=False)["input_ids"]
        answer_ids = tok(answer, add_special_tokens=False)["input_ids"]
        return {"input_ids": prompt_ids + answer_ids,
                "labels": [-100] * len(prompt_ids) + answer_ids}


def sft_collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    input_ids, labels, attn = [], [], []
    for b in batch:
        pad = tok.pad_token_id
        n = maxlen - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [pad] * n)
        labels.append(b["labels"] + [-100] * n)
        attn.append([1] * len(b["input_ids"]) + [0] * n)
    return {"input_ids": torch.tensor(input_ids), "labels": torch.tensor(labels),
            "attention_mask": torch.tensor(attn)}


loader = DataLoader(SftDataset(pairs), batch_size=SFT_BATCH, shuffle=True,
                    collate_fn=sft_collate)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
t0 = time.time()
losses = []
model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    n = 0
    for batch in loader:
        with torch.autocast("cuda", dtype=dtype):
            loss = model(
                input_ids=batch["input_ids"].cuda(),
                attention_mask=batch["attention_mask"].cuda(),
                labels=batch["labels"].cuda(),
            ).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n += 1
    losses.append(round(epoch_loss / n, 4))
    print(f"epoch {epoch}: mean loss {losses[-1]}")
sft_s = round(time.time() - t0, 1)
check("SFT ran", all(math.isfinite(l) for l in losses) and len(losses) == EPOCHS,
      f"epoch losses={losses} ({sft_s}s)")
model.save_pretrained(os.path.join(WORK, "qwen-sft-lora"))
tok.save_pretrained(os.path.join(WORK, "qwen-sft-lora"))
trainable, total_params = model.get_nb_trainable_parameters()
REPORT["sft"] = {"targets": targets, "epoch_losses": losses, "seconds": sft_s,
                 "trainable_params": trainable, "total_params": total_params}
print(f"adapter saved to {WORK}/qwen-sft-lora")

# Post-SFT re-check on the same frozen bank.
model.eval()
t0 = time.time()
sft_outputs = generate_batch(REQUESTS)
sft_gen_s = round(time.time() - t0, 1)
post = contract_results(REQUESTS, sft_outputs)
REPORT["post_sft"] = {k: post[k] for k in ("pass_json", "pass_once", "total", "rate_json", "rate_once")}
print(f"post-SFT: JSON {post['rate_json']:.0%}, name-exactly-once {post['rate_once']:.0%} ({sft_gen_s}s)")
check("SFT improved or held the contract rate",
      post["rate_once"] >= zero["rate_once"],
      f"zero-shot once-rate={zero['rate_once']} post-SFT once-rate={post['rate_once']}")

In [ ]:
REPORT["finished_utc"] = datetime.datetime.now(datetime.UTC).isoformat()
REPORT["zeroshot_failures"] = zero["failures"]
REPORT["post_sft_failures"] = post["failures"]

report_path = os.path.join(WORK, "m05_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(REPORT, f, indent=2, ensure_ascii=False)

print("=" * 70)
print("M05 GENERATOR SUMMARY")
print("=" * 70)
for name, res in REPORT["checks"].items():
    print(f"[{'PASS' if res['ok'] else 'FAIL'}] {name}")
    if res["detail"]:
        print(f"       {res['detail']}")
print("=" * 70)
print(f"adapter: {WORK}/qwen-sft-lora")
print(f"report:  {report_path}")
print("Next: bring back m05_report.json; the SFT'd adapter is the M06 ordinary-generation")
print("baseline generator, pending owner review of the failure examples.")
if not all(r["ok"] for r in REPORT["checks"].values()):
    print("NOTE: some checks FAILED - record them honestly; do not retune silently.")